# 🧭 Strategy Guide: Tackling Linear-Regression Projects on Lending / Transactional Data
A general-purpose playbook — distilled from the bank-loan pricing project — for
approaching *any* new administrative/transactional dataset (loans, claims, orders,
subscriptions) with a linear regression lens.

## The Big Picture: A 5-Phase Framework

```
PHASE 1: FRAME        -> What question, for whom, using what outcome?
PHASE 2: INTERROGATE   -> Understand the data before touching a model
PHASE 3: PREPARE       -> Clean, filter, transform, split
PHASE 4: MODEL         -> Simple -> assess -> multiple -> assess
PHASE 5: COMMUNICATE   -> Interpret honestly, state limits, summarize
```

This is the same framework used for the PSID earnings project — it generalizes across
domains. What changes from dataset to dataset is *what to look for* in each phase; this
guide highlights the lending-specific version of each checklist.

---
## Phase 1 — Frame the Question

1. **The outcome variable** — pricing (`interest_rate`, `APR`)? Approved amount?
   Time-to-close? Each implies a different modeling approach and audience.
2. **The unit of observation** — one row per loan application? Per funded loan? Per
   borrower (who may have multiple loans)? Get this wrong and your "n" is meaningless.
3. **Pre- vs. post-origination variables.** Lending/transactional data is full of
   variables measured *after* the outcome you're modeling (e.g., `loan_status`,
   `days_to_default`, `total_payments_received`). Using these as predictors of a
   pricing/approval decision that happened *before* them is a subtle but serious
   **look-ahead / target-leakage** error.
4. **A hypothesis or two** — e.g., "I expect higher credit scores to earn lower rates."

> 🎯 **Rule of thumb:** for any candidate predictor, ask "could this have been known
> *at the moment* the outcome was determined?" If not, it doesn't belong in the model.

---
## Phase 2 — Interrogate the Data

### Checklist (lending/transactional flavor)
- [ ] Check for **exact duplicate records** — resubmitted applications, re-synced ETL
      batches, and joined tables are common sources of duplication in operational data
- [ ] Check every numeric column's min/max for **impossible values** given its real-world
      meaning (a credit score of 0, an age of 999, a rate of 999%)
- [ ] Check whether a **ratio or derived column** (DTI, LTV, utilization %) can only be
      computed when its inputs are present — and whether the system defaults to 0/NaN
      when it can't
- [ ] Check for **post-outcome variables** that shouldn't be used as predictors (see
      Phase 1, point 3)
- [ ] Check categorical columns for **inconsistent encoding** (`"NY"` vs `"New York"`,
      trailing whitespace, mixed case) — common when data comes from multiple systems

### Decision guide: "Is this ratio column's zero real or structural?"
```
Does the ratio require another column (income, balance, limit) to compute?
        |
        +-- YES --> Cross-tab the ratio's zeros against that other column's
        |           missingness. If they line up closely, the zero is
        |           STRUCTURAL (a computation artifact) -- null it out.
        |
        +-- NO, it's directly measured --> A zero is more likely genuine
                    (e.g., "number of late payments = 0" really can mean none).
```

### 🚩 Red flags to catch here
- A rate/percentage column with values far outside its plausible range (a "system
  glitch" sentinel, not a real observation)
- A derived ratio that is suspiciously often exactly 0 or exactly 100
- A predictor that is only ever populated *after* the outcome occurred
- Row counts that don't match a known business total (hints at hidden duplicates or
  missing joins)

---
## Phase 3 — Prepare the Data

### Checklist
- [ ] De-duplicate first, before any other cleaning step (duplicates otherwise get
      "cleaned" twice and silently double-weighted in your model)
- [ ] Replace sentinel codes with `NaN`
- [ ] Resolve structural zeros/ratios per the Phase 2 decision guide
- [ ] Drop or impute rows missing required fields — and **write down who you're
      excluding** (e.g., "thin-file applicants," "undisclosed-income applicants") so you
      can caveat generalizability later
- [ ] Check skew on **every** continuous variable, predictors included — lending amounts
      and incomes are almost always right-skewed and benefit from a log transform, while
      rate/percentage outcomes usually don't need one
- [ ] Trim genuine statistical outliers (IQR/boxplot) **after** structural issues are
      already resolved, so you don't accidentally "fix" a data quality problem by
      mislabeling it a statistical outlier
- [ ] Split into train/test before evaluating any model's fit

### Decision guide: "Should I log-transform this variable?"
```
Is it a dollar amount, count, duration, or similarly unbounded positive quantity
with a long right tail?
        |
        +-- YES --> log-transform is usually a good default
        |
        +-- Is it a percentage/rate/ratio with a naturally bounded range
            (0-100%, a score out of 850, etc.)?
                |
                +-- YES --> usually leave it untransformed; a bounded
                            outcome rarely needs a log transform
```

### 🚩 Red flags to catch here
- Filtering "outliers" using bounds computed on the un-deduplicated or un-cleaned data
- Silently imputing a ratio's missing denominator with an arbitrary constant instead of
  flagging the row as missing
- Log-transforming a bounded rate/percentage outcome out of habit

---
## Phase 4 — Model, Assess, Extend

### The core loop (identical to any linear regression project)
```
1. FIT       -- smf.ols('y ~ x1 + x2 + ...', data=train).fit()
2. FIT STATS -- R², adjusted R², RSE
3. RESIDUALS -- residuals vs fitted, Q-Q plot -- do assumptions hold?
4. VISUALIZE -- plot the fit against the data (and a flexible LOWESS for comparison)
5. INTERPRET -- coefficients, p-values, CIs -- in units your business audience understands
6. PREDICT   -- evaluate on the held-out TEST set, compare to a naive baseline
```

### Decision guide: "Which predictors belong in a pricing/underwriting model?"
```
Was this value known BEFORE the pricing/underwriting decision was made?
        |
        +-- NO (e.g., loan_status, days_late, total_paid) --> EXCLUDE.
        |    These belong in a separate, later-stage model (e.g., predicting default
        |    risk on already-originated loans), not in a pricing model.
        |
        +-- YES --> Does domain knowledge suggest it plausibly affects pricing
                    (risk, size, term, verification status)?
                        |
                        +-- YES --> include, then check adjusted R² improvement,
                                    nested F-test significance, and VIF
                        +-- NO / unsure --> consider leaving out, or flag as a
                                    candidate for a follow-up analysis
```

### 🚩 Red flags to catch here
- Including a post-decision variable and getting suspiciously high R² (classic sign of
  target leakage)
- Interpreting a categorical coefficient without naming its reference category
- Reporting only training-set fit statistics, never test-set ones

---
## Phase 5 — Communicate Honestly

### Checklist
- [ ] State clearly who the analytic sample represents (e.g., "applicants with fully
      verifiable income and credit history") and who was excluded
- [ ] Flag explicitly that this is **simulated/illustrative** data if it is — never let a
      demo dataset's specific numbers be mistaken for real institutional policy
- [ ] Frame coefficients as associations from historical pricing data, not as proof that
      changing one input would causally change future prices
- [ ] Report effect sizes in real business units (percentage points, dollars) alongside
      p-values
- [ ] Suggest the natural adjacent project (e.g., a default-risk model using
      `loan_status`) as a next step rather than trying to cram every question into one
      analysis

### A reusable "Limitations" template
> "This analysis covers [N] loans meeting [inclusion criteria]. Applicants who
> [excluded group] were not included and results should not be assumed to generalize to
> them. Because this reflects historical pricing/underwriting decisions rather than a
> controlled experiment, coefficients describe association, not a guaranteed causal
> effect of changing any one input. [Known limitation, e.g., simulated data / single time
> period] limits how confidently these specific figures should be used. A natural next
> step would be [a default-risk model / multiple vintages / additional risk features]."

### 🚩 Red flags to catch here
- Presenting a demo/simulated model's coefficients as real institutional rate-setting
  policy
- Mixing a pricing question and a default-prediction question into one model without
  flagging the target-leakage risk

---
## Quick-Start Template for a New Project

```
1. Outcome variable: ____________________  (continuous? bounded/rate, or unbounded/skewed?)
2. Unit of observation: ____________________  (application? loan? borrower? transaction?)
3. Decision point in time: ____________________  (what moment does "before/after" split on?)
4. Candidate predictors (known BEFORE the decision point, with expected sign):
   - ____________________  (expect: + / - / unsure)
   - ____________________  (expect: + / - / unsure)
5. Known data quirks to check for (dup records, sentinel codes, structural
   ratio zeros, post-decision leakage columns):
   - ____________________
6. What would a "good enough" model look like for this audience?
```

Then walk the 5-phase framework above, using `04_reusable_template.ipynb` for the actual
code scaffolding and `03_project_cheatsheet.ipynb` for syntax lookups along the way.